# NUS ST4248 — Advanced Ensembles, Stacking and SVM Hyperparameter Lab

This notebook is a focused, experiment-driven continuation of **Statistical Learning II**.

It studies:

1. **Gradient Boosting parameter effects**
   - `loss`
   - `learning_rate`
   - `n_estimators`
   - `subsample`
   - `max_depth`
   - `min_impurity_decrease`
   - `max_features`
   - `max_leaf_nodes`

2. **Boosting-family comparison**
   - Gradient Boosting
   - AdaBoost
   - Histogram Gradient Boosting
   - performance tables and decision-boundary visualisations

3. **Random Forest parameter effects**
   - ensemble size
   - depth
   - split / leaf controls
   - feature subsampling
   - bootstrap sample size
   - impurity threshold
   - maximum leaf nodes

4. **Stacking**
   - heterogeneous base learners
   - out-of-fold meta-features
   - meta-learning

5. **Support Vector Machines beyond $C$**
   - linear, polynomial, RBF and sigmoid kernels
   - $\gamma$
   - degree
   - `coef0`
   - shrinking heuristic
   - optimisation tolerance

The central question is:

$$
\boxed{
\text{How does each hyperparameter alter capacity, bias, variance, computation and generalisation?}
}
$$

## Experimental design

We use two datasets:

- **Breast Cancer Wisconsin** for realistic multivariate performance evaluation.
- **Two Moons** for visualising nonlinear decision boundaries.

Most parameter studies are **one-at-a-time sweeps** under stratified cross-validation. This makes mechanisms easy to study, but parameters can interact, so one-at-a-time sweeps are not a substitute for joint tuning.

We report:

- ROC AUC
- F1
- balanced accuracy
- fit time

Bokeh is used for plots, while ordinary HTML is used for result tables so tables remain stable when the notebook is saved and reopened.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from IPython.display import display, HTML

from sklearn.base import clone
from sklearn.datasets import load_breast_cancer, make_moons
from sklearn.ensemble import (
    GradientBoostingClassifier,
    AdaBoostClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
    StackingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    log_loss,
)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from bokeh.io import output_notebook, show
from bokeh.models import LinearColorMapper, ColorBar
from bokeh.palettes import Category10, Viridis256
from bokeh.plotting import figure

output_notebook()

RANDOM_STATE = 42
FAST_MODE = True

CV_SPLITS = 3 if FAST_MODE else 5
DEFAULT_RF_TREES = 220 if FAST_MODE else 500
DEFAULT_GB_TREES = 180 if FAST_MODE else 350

print(f"FAST_MODE={FAST_MODE}, CV_SPLITS={CV_SPLITS}")

Loading BokehJS ...

FAST_MODE=True, CV_SPLITS=3


In [2]:
def show_table(df, title="Table", height=320, digits=4):
    x = df.copy()
    x.columns = [str(c) for c in x.columns]

    for c in x.select_dtypes(include=np.number).columns:
        x[c] = x[c].round(digits)

    table_html = x.to_html(index=False, border=0, classes="lab-table")

    display(HTML(f'''
    <div style="max-width:1100px;margin:8px 0 18px 0;font-family:Arial,sans-serif;">
      <div style="font-size:15px;font-weight:700;margin-bottom:8px;">{title}</div>
      <div style="max-height:{height}px;overflow:auto;border:1px solid #ddd;border-radius:6px;">
        <style>
          .lab-table {{width:100%;border-collapse:collapse;font-size:13px;}}
          .lab-table thead th {{
              position:sticky;top:0;background:#f4f4f4;padding:8px 10px;
              border-bottom:2px solid #ccc;text-align:left;
          }}
          .lab-table tbody td {{
              padding:7px 10px;border-bottom:1px solid #eee;text-align:left;
          }}
          .lab-table tbody tr:nth-child(even) {{background:#fafafa;}}
        </style>
        {table_html}
      </div>
    </div>
    '''))


DASHES = ["solid", "dashed", "dotdash"]


def plot_sweep(df, title, x_label, categorical=False):
    metrics = [
        ("roc_auc_mean", "ROC AUC"),
        ("f1_mean", "F1"),
        ("balanced_accuracy_mean", "Balanced accuracy"),
    ]
    palette = Category10[10]

    if categorical:
        xvals = [str(v) for v in df["value"]]
        p = figure(
            x_range=xvals, width=860, height=400, title=title,
            x_axis_label=x_label, y_axis_label="cross-validated score"
        )
    else:
        xvals = pd.to_numeric(df["value"])
        p = figure(
            width=860, height=400, title=title,
            x_axis_label=x_label, y_axis_label="cross-validated score"
        )

    for i, (col, label) in enumerate(metrics):
        p.line(
            xvals, df[col],
            line_width=2.8,
            line_dash=DASHES[i],
            color=palette[i],
            legend_label=label,
        )
        p.scatter(xvals, df[col], size=7, color=palette[i])

    p.legend.location = "bottom_right"
    p.legend.click_policy = "hide"
    show(p)

    if categorical:
        p2 = figure(
            x_range=xvals, width=860, height=280,
            title=f"{title} — fit time",
            x_axis_label=x_label, y_axis_label="mean fit time (s)"
        )
    else:
        p2 = figure(
            width=860, height=280,
            title=f"{title} — fit time",
            x_axis_label=x_label, y_axis_label="mean fit time (s)"
        )

    p2.line(xvals, df["fit_time_mean"], line_width=2.8, color=palette[3])
    p2.scatter(xvals, df["fit_time_mean"], size=7, color=palette[3])
    show(p2)

# Part I — Datasets and evaluation machinery

## Real multivariate dataset

The Breast Cancer Wisconsin dataset has 30 numerical predictors and a binary target. It is small enough for repeated cross-validation yet rich enough for meaningful nonlinear modelling.

## Two-dimensional nonlinear dataset

The two-moons dataset is used only for **geometric intuition**. It lets us see how different algorithms carve up feature space.

In [3]:
cancer = load_breast_cancer(as_frame=True)
X = cancer.data.copy()
y = cancer.target.copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_STATE,
)

X2, y2 = make_moons(
    n_samples=700,
    noise=0.23,
    random_state=RANDOM_STATE,
)

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2,
    test_size=0.30,
    stratify=y2,
    random_state=RANDOM_STATE,
)

print("Real dataset:", X.shape)
print("Train/test:", X_train.shape, X_test.shape)
print("Two-moons:", X2.shape)

Real dataset: (569, 30)
Train/test: (426, 30) (143, 30)
Two-moons: (700, 2)


In [47]:
p = figure(
    width=760, height=430,
    title="Two-moons dataset",
    x_axis_label="x1", y_axis_label="x2"
)
p.scatter(
    X2[y2 == 0, 0], X2[y2 == 0, 1],
    size=6, alpha=0.55, legend_label="class 0"
)
p.scatter(
    X2[y2 == 1, 0], X2[y2 == 1, 1],
    size=6, alpha=0.55, marker="triangle", legend_label="class 1"
)
p.legend.location = "top_right"
show(p)

In [5]:
cv = StratifiedKFold(
    n_splits=CV_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

SCORING = {
    "roc_auc": "roc_auc",
    "f1": "f1",
    "balanced_accuracy": "balanced_accuracy",
}


def cv_evaluate(model, X_data=X_train, y_data=y_train):
    scores = cross_validate(
        model,
        X_data,
        y_data,
        cv=cv,
        scoring=SCORING,
        n_jobs=-1,
        return_train_score=False,
    )
    return {
        "roc_auc_mean": scores["test_roc_auc"].mean(),
        "roc_auc_sd": scores["test_roc_auc"].std(ddof=1),
        "f1_mean": scores["test_f1"].mean(),
        "balanced_accuracy_mean": scores["test_balanced_accuracy"].mean(),
        "fit_time_mean": scores["fit_time"].mean(),
    }


def run_sweep(base_model, param_name, values):
    rows = []
    for value in values:
        model = clone(base_model).set_params(**{param_name: value})
        rows.append({
            "parameter": param_name,
            "value": value,
            **cv_evaluate(model),
        })
    return pd.DataFrame(rows)


def holdout_metrics(model):
    fitted = clone(model).fit(X_train, y_train)
    pred = fitted.predict(X_test)
    prob = fitted.predict_proba(X_test)[:, 1]

    return {
        "accuracy": accuracy_score(y_test, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, prob),
        "log_loss": log_loss(y_test, prob),
    }

In [48]:
def show_decision_boundary(model, title):
    fitted = clone(model).fit(X2_train, y2_train)

    x_min, x_max = X2_train[:, 0].min() - 0.6, X2_train[:, 0].max() + 0.6
    y_min, y_max = X2_train[:, 1].min() - 0.6, X2_train[:, 1].max() + 0.6

    nx = ny = 240
    gx = np.linspace(x_min, x_max, nx)
    gy = np.linspace(y_min, y_max, ny)
    xx, yy = np.meshgrid(gx, gy)
    grid = np.column_stack([xx.ravel(), yy.ravel()])

    if hasattr(fitted, "predict_proba"):
        z = fitted.predict_proba(grid)[:, 1].reshape(ny, nx)
    else:
        score = fitted.decision_function(grid)
        z = (1 / (1 + np.exp(-score))).reshape(ny, nx)

    mapper = LinearColorMapper(
        palette=Viridis256,
        low=0,
        high=1,
    )

    p = figure(
        width=760, height=460,
        title=title,
        x_axis_label="x1", y_axis_label="x2"
    )

    p.image(
        image=[z],
        x=x_min, y=y_min,
        dw=x_max-x_min,
        dh=y_max-y_min,
        color_mapper=mapper,
        alpha=0.55,
    )

    p.scatter(
        X2_train[y2_train == 0, 0],
        X2_train[y2_train == 0, 1],
        size=6, alpha=0.75,
        legend_label="class 0",
    )
    p.scatter(
        X2_train[y2_train == 1, 0],
        X2_train[y2_train == 1, 1],
        size=6, alpha=0.75,
        marker="triangle",
        legend_label="class 1",
    )

    p.add_layout(ColorBar(color_mapper=mapper, title="P(class=1)"), "right")
    p.legend.location = "top_right"
    show(p)

    pred = fitted.predict(X2_test)
    prob = fitted.predict_proba(X2_test)[:, 1]
    print(
        f"Two-moons test accuracy={accuracy_score(y2_test, pred):.3f}; "
        f"ROC AUC={roc_auc_score(y2_test, prob):.3f}"
    )

# Part II — Gradient Boosting hyperparameter laboratory

Gradient Boosting constructs a sequential additive model

$$
F_M(x)=F_0(x)+\sum_{m=1}^{M}\eta h_m(x),
$$

where each $h_m$ is usually a shallow decision tree.

The important distinction is that some parameters control the **ensemble**, while others control the **individual trees**.

### Ensemble-level controls

- `loss`
- `learning_rate`
- `n_estimators`
- `subsample`

### Tree-level controls

- `max_depth`
- `min_impurity_decrease`
- `max_features`
- `max_leaf_nodes`

The learner's effective complexity is produced by the interaction of both levels.

In [49]:
gb_base = GradientBoostingClassifier(
    loss="log_loss",
    learning_rate=0.05,
    n_estimators=DEFAULT_GB_TREES,
    subsample=1.0,
    max_depth=2,
    min_impurity_decrease=0.0,
    max_features=None,
    max_leaf_nodes=None,
    random_state=RANDOM_STATE,
)
gb_base

GradientBoostingClassifier(learning_rate=0.05, max_depth=2, n_estimators=180,
                           random_state=42)

## 1. Loss function

For binary classification, two useful choices are:

### `log_loss`

Optimises probabilistic cross-entropy / deviance.

### `exponential`

Uses an exponential-loss formulation related to AdaBoost.

Changing the loss changes what kind of residual-like signal later trees are correcting.

In [50]:
gb_loss_df = run_sweep(gb_base, "loss", ["log_loss", "exponential"])
show_table(gb_loss_df, "Gradient Boosting — loss function")
plot_sweep(
    gb_loss_df,
    "Gradient Boosting: loss function",
    "loss",
    categorical=True,
)

parameter,value,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean
loss,log_loss,0.9876,0.0071,0.9571,0.9417,0.6818
loss,exponential,0.9865,0.0087,0.9552,0.9398,0.6637


## 2. Learning rate

The learning rate $\eta$ shrinks the contribution of every new tree:

$$
F_m(x)=F_{m-1}(x)+\eta h_m(x).
$$

Smaller values make each correction more conservative. They usually require more boosting stages, but can improve generalisation.

This gives the classic interaction

$$
\boxed{\text{learning rate}\times\text{number of estimators}}.
$$

In [9]:
gb_lr_df = run_sweep(
    gb_base,
    "learning_rate",
    [0.01, 0.03, 0.07, 0.15] if FAST_MODE else [0.005, 0.01, 0.03, 0.05, 0.10, 0.20],
)
show_table(gb_lr_df, "Gradient Boosting — learning_rate")
plot_sweep(gb_lr_df, "Gradient Boosting: learning rate", "learning_rate")

parameter,value,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean
learning_rate,0.01,0.9790,0.0120,0.9607,0.9480,0.5949
learning_rate,0.03,0.9850,0.0088,0.9531,0.9379,0.5832
learning_rate,0.07,0.9880,0.0071,0.9532,0.9379,0.5863
learning_rate,0.15,0.9891,0.0060,0.9663,0.9548,0.5499


## 3. Number of estimators

`n_estimators` is the number of sequential boosting stages.

Unlike Random Forest, adding trees does not merely stabilise an average. Each stage changes the current fitted function.

Too few stages can underfit; too many can eventually overfit, particularly when the learning rate or base trees are aggressive.

In [52]:
gb_n_df = run_sweep(
    gb_base,
    "n_estimators",
    [40, 100, 220, 400] if FAST_MODE else [40, 80, 150, 300, 600],
)
show_table(gb_n_df, "Gradient Boosting — n_estimators")
plot_sweep(gb_n_df, "Gradient Boosting: number of estimators", "n_estimators")

parameter,value,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean
n_estimators,40,0.9789,0.0121,0.9589,0.9448,0.1384
n_estimators,100,0.9866,0.0068,0.9549,0.9411,0.3461
n_estimators,220,0.9881,0.0071,0.9590,0.9435,0.7781
n_estimators,400,0.9881,0.0074,0.9645,0.9517,1.3138


## 4. Subsample — stochastic gradient boosting

When `subsample < 1`, each stage is trained on a random subset of the observations.

This can:

- inject useful randomness,
- reduce variance,
- weaken each individual correction,
- sometimes improve generalisation,
- eventually hurt if too little data are shown to each stage.

In [11]:
gb_sub_df = run_sweep(gb_base, "subsample", [0.4, 0.6, 0.8, 1.0])
show_table(gb_sub_df, "Gradient Boosting — subsample")
plot_sweep(gb_sub_df, "Gradient Boosting: subsample", "subsample")

parameter,value,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean
subsample,0.4,0.9905,0.0078,0.9724,0.9579,0.4922
subsample,0.6,0.9903,0.0066,0.9685,0.9542,0.4133
subsample,0.8,0.9886,0.0071,0.9609,0.9454,0.5474
subsample,1.0,0.9876,0.0071,0.9571,0.9417,0.5568


## 5. Maximum depth

Depth controls the complexity of each weak learner.

- depth 1 → decision stump
- depth 2–3 → limited interactions
- deeper trees → increasingly complex residual corrections

Boosting often works well with shallow trees precisely because the ensemble accumulates complexity over many stages.

In [12]:
gb_depth_df = run_sweep(gb_base, "max_depth", [1, 2, 3, 4, 6])
show_table(gb_depth_df, "Gradient Boosting — max_depth")
plot_sweep(gb_depth_df, "Gradient Boosting: max_depth", "max_depth")

parameter,value,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean
max_depth,1,0.9866,0.0100,0.9611,0.9441,0.3573
max_depth,2,0.9876,0.0071,0.9571,0.9417,0.5457
max_depth,3,0.9850,0.0096,0.9590,0.9435,0.7973
max_depth,4,0.9664,0.0352,0.9519,0.9335,0.9942
max_depth,6,0.9325,0.0241,0.9326,0.9097,1.1574


## 6. `min_impurity_decrease`

A split is allowed only if it decreases weighted impurity by at least a threshold.

If

$$
\Delta I < \tau,
$$

the split is rejected.

Increasing the threshold therefore acts as **tree-level regularisation**:

- weak splits disappear,
- trees become simpler,
- variance can fall,
- excessive thresholds cause underfitting.

In [53]:
gb_imp_df = run_sweep(
    gb_base,
    "min_impurity_decrease",
    [0.0, 1e-5, 1e-4, 1e-3, 1e-2],
)
show_table(gb_imp_df, "Gradient Boosting — min_impurity_decrease")
plot_sweep(
    gb_imp_df,
    "Gradient Boosting: min_impurity_decrease",
    "min_impurity_decrease",
)

parameter,value,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean
min_impurity_decrease,0.0000,0.9876,0.0071,0.9571,0.9417,0.5904
min_impurity_decrease,0.0000,0.9875,0.0072,0.9571,0.9417,0.6365
min_impurity_decrease,0.0001,0.9875,0.0072,0.9571,0.9417,0.6145
min_impurity_decrease,0.0010,0.9876,0.0071,0.9571,0.9417,0.5837
min_impurity_decrease,0.0100,0.9872,0.0073,0.9551,0.9398,0.5605


## 7. `max_features`

This restricts the number of predictors considered at each split.

Possible choices include:

- `None` → all predictors
- `"sqrt"`
- `"log2"`
- a fraction such as `0.5`

Reducing the available features adds randomness to each boosting tree. This can reduce variance, but it can also hide an important corrective feature from a stage.

In [14]:
gb_mf_df = run_sweep(
    gb_base,
    "max_features",
    [None, "sqrt", "log2", 0.5],
)
show_table(gb_mf_df, "Gradient Boosting — max_features")
plot_sweep(
    gb_mf_df,
    "Gradient Boosting: max_features",
    "max_features",
    categorical=True,
)

parameter,value,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean
max_features,None,0.9876,0.0071,0.9571,0.9417,0.6160
max_features,sqrt,0.9891,0.0064,0.9592,0.9423,0.2764
max_features,log2,0.9890,0.0068,0.9609,0.9454,0.2390
max_features,0.5,0.9878,0.0074,0.9570,0.9417,0.3727


## 8. `max_leaf_nodes`

`max_leaf_nodes` directly caps the number of terminal regions in each tree.

This differs from depth:

- depth limits the longest path,
- leaf count limits the total number of terminal regions.

Increasing the leaf count increases the interaction and partition complexity available to each boosting stage.

In [15]:
gb_leaf_df = run_sweep(
    gb_base,
    "max_leaf_nodes",
    [2, 4, 8, 16, 32],
)
show_table(gb_leaf_df, "Gradient Boosting — max_leaf_nodes")
plot_sweep(
    gb_leaf_df,
    "Gradient Boosting: max_leaf_nodes",
    "max_leaf_nodes",
)

parameter,value,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean
max_leaf_nodes,2,0.9866,0.0100,0.9611,0.9441,0.6238
max_leaf_nodes,4,0.9876,0.0071,0.9571,0.9417,0.6644
max_leaf_nodes,8,0.9876,0.0071,0.9571,0.9417,0.6547
max_leaf_nodes,16,0.9876,0.0071,0.9571,0.9417,0.5398
max_leaf_nodes,32,0.9876,0.0071,0.9571,0.9417,0.5737


## 9. Joint interaction: learning rate × estimator count

One-at-a-time sweeps can hide parameter interactions.

For boosting, the most important interaction is often between step size and number of stages.

In [16]:
interaction_rows = []

for lr in [0.02, 0.05, 0.10, 0.20]:
    for n_est in ([60, 120, 240, 420] if FAST_MODE else [80, 160, 320, 640]):
        model = clone(gb_base).set_params(
            learning_rate=lr,
            n_estimators=n_est,
        )
        interaction_rows.append({
            "learning_rate": lr,
            "n_estimators": n_est,
            **cv_evaluate(model),
        })

gb_interaction_df = pd.DataFrame(interaction_rows)

show_table(
    gb_interaction_df.sort_values("roc_auc_mean", ascending=False),
    "Gradient Boosting — learning_rate × n_estimators",
    height=380,
)

learning_rate,n_estimators,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean
0.20,420,0.9905,0.0048,0.9589,0.9435,1.4031
0.20,240,0.9903,0.0050,0.9589,0.9435,1.1388
0.20,120,0.9901,0.0048,0.9627,0.9486,0.4568
0.10,420,0.9887,0.0065,0.9550,0.9398,1.6855
0.05,240,0.9886,0.0068,0.9608,0.9467,0.8373
0.20,60,0.9884,0.0052,0.9590,0.9435,0.1936
0.10,240,0.9883,0.0071,0.9625,0.9498,0.8338
0.10,120,0.9883,0.0066,0.9570,0.9417,0.4064
0.05,420,0.9880,0.0077,0.9645,0.9517,1.3619
0.02,420,0.9871,0.0071,0.9551,0.9398,1.4920


In [17]:
pivot = gb_interaction_df.pivot(
    index="learning_rate",
    columns="n_estimators",
    values="roc_auc_mean",
)

matrix = pivot.values
mapper = LinearColorMapper(
    palette=Viridis256,
    low=float(matrix.min()),
    high=float(matrix.max()),
)

p = figure(
    x_range=[str(v) for v in pivot.columns],
    y_range=[str(v) for v in pivot.index],
    width=760,
    height=430,
    title="Gradient Boosting: learning rate × estimators — CV ROC AUC",
    x_axis_label="n_estimators",
    y_axis_label="learning_rate",
)

for lr in pivot.index:
    for n_est in pivot.columns:
        value = float(pivot.loc[lr, n_est])
        source = {"x": [str(n_est)], "y": [str(lr)], "value": [value]}
        p.rect(
            x="x", y="y", width=1, height=1,
            source=source,
            fill_color={"field": "value", "transform": mapper},
            line_color=None,
        )
        p.text(
            x=[str(n_est)], y=[str(lr)],
            text=[f"{value:.3f}"],
            text_align="center",
            text_baseline="middle",
        )

p.add_layout(ColorBar(color_mapper=mapper), "right")
show(p)

# Part III — Gradient Boosting vs AdaBoost vs Histogram Gradient Boosting

Although all three are called boosting, they optimise differently.

## Gradient Boosting

Sequentially fits trees to negative gradients of a loss function.

## AdaBoost

Increases emphasis on observations that previous weak learners misclassified.

A simplified intuition is:

$$
w_i^{(m+1)}
\propto
w_i^{(m)}
\exp\left(\alpha_m I[y_i\neq h_m(x_i)]\right).
$$

## Histogram Gradient Boosting

Bins continuous features and searches for splits using histograms rather than sorting every raw value repeatedly.

This can make training substantially more efficient on larger datasets.

In [18]:
boost_models = {
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=DEFAULT_GB_TREES,
        learning_rate=0.05,
        max_depth=2,
        random_state=RANDOM_STATE,
    ),
    "AdaBoost": AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=1,
            random_state=RANDOM_STATE,
        ),
        n_estimators=DEFAULT_GB_TREES,
        learning_rate=0.05,
        random_state=RANDOM_STATE,
    ),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=DEFAULT_GB_TREES,
        max_leaf_nodes=15,
        l2_regularization=0.1,
        random_state=RANDOM_STATE,
    ),
}

rows = []
for name, model in boost_models.items():
    cv_result = cv_evaluate(model)
    holdout = holdout_metrics(model)
    rows.append({
        "model": name,
        **cv_result,
        "holdout_accuracy": holdout["accuracy"],
        "holdout_f1": holdout["f1"],
        "holdout_roc_auc": holdout["roc_auc"],
        "holdout_log_loss": holdout["log_loss"],
    })

boost_compare_df = (
    pd.DataFrame(rows)
    .sort_values("holdout_roc_auc", ascending=False)
    .reset_index(drop=True)
)

show_table(
    boost_compare_df,
    "Boosting-family performance comparison",
    height=260,
)

model,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean,holdout_accuracy,holdout_f1,holdout_roc_auc,holdout_log_loss
Hist Gradient Boosting,0.9883,0.0084,0.9644,0.9530,0.2178,0.958,0.9670,0.9937,0.0986
Gradient Boosting,0.9876,0.0071,0.9571,0.9417,0.6625,0.951,0.9617,0.9933,0.1040
AdaBoost,0.9864,0.0079,0.9572,0.9404,0.5529,0.958,0.9674,0.9912,0.2911


## Decision boundaries

The two-moons dataset exposes **how** each boosting family reaches its score.

Look for:

- smooth versus fragmented transitions,
- isolated local corrections,
- whether noise bends the boundary,
- confidence transitions between classes.

In [19]:
for name, model in boost_models.items():
    show_decision_boundary(
        model,
        f"{name} — decision probability surface",
    )

Two-moons test accuracy=0.933; ROC AUC=0.987


Two-moons test accuracy=0.905; ROC AUC=0.970


Two-moons test accuracy=0.919; ROC AUC=0.981


# Part IV — Random Forest parameter laboratory

Random Forest combines:

1. bootstrap sampling of observations,
2. feature subsampling at each split,
3. averaging across many trees.

Its behaviour depends on both **individual-tree strength** and **tree-to-tree diversity**.

A useful statistical intuition is:

$$
\text{ensemble quality}
\approx
\text{strong trees}
+
\text{low correlation among tree errors}.
$$

In [20]:
rf_base = RandomForestClassifier(
    n_estimators=DEFAULT_RF_TREES,
    criterion="gini",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    bootstrap=True,
    max_samples=None,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

rf_base

RandomForestClassifier(n_estimators=220, n_jobs=-1, random_state=42)

## 1. Number of trees

Increasing `n_estimators` mainly stabilises the ensemble average.

Unlike boosting, extra trees are not sequential residual corrections.

Performance often reaches a plateau, while fit time and memory continue growing.

In [21]:
rf_n_df = run_sweep(
    rf_base,
    "n_estimators",
    [40, 100, 250, 450] if FAST_MODE else [40, 100, 250, 500, 800],
)
show_table(rf_n_df, "Random Forest — n_estimators")
plot_sweep(rf_n_df, "Random Forest: number of trees", "n_estimators")

parameter,value,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean
n_estimators,40,0.9882,0.0109,0.9590,0.9448,0.1715
n_estimators,100,0.9873,0.0111,0.9572,0.9417,0.4156
n_estimators,250,0.9875,0.0118,0.9591,0.9435,1.0058
n_estimators,450,0.9872,0.0122,0.9609,0.9467,1.7814


## 2. Maximum depth

Deeper trees lower individual-tree bias but increase variance.

Random Forest can often tolerate deep trees because averaging suppresses some of that variance. However, unrestricted trees can still increase computation and create highly sample-specific leaves.

In [54]:
rf_depth_df = run_sweep(rf_base, "max_depth", [2, 4, 6, 10, None])
show_table(rf_depth_df, "Random Forest — max_depth")
plot_sweep(
    rf_depth_df,
    "Random Forest: maximum depth",
    "max_depth",
    categorical=True,
)

parameter,value,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean
max_depth,2.0,0.9852,0.0132,0.9595,0.9397,0.9181
max_depth,4.0,0.9886,0.0087,0.9648,0.9504,1.0424
max_depth,6.0,0.9868,0.0125,0.9572,0.9404,0.9649
max_depth,10.0,0.9878,0.0116,0.9591,0.9435,0.8809
max_depth,NaN,0.9878,0.0116,0.9591,0.9435,0.9395


## 3. `min_samples_split`

A node cannot split unless it contains at least this many observations.

Larger values:

- suppress small-node splits,
- reduce tree depth indirectly,
- raise bias,
- reduce variance.

In [55]:
rf_split_df = run_sweep(
    rf_base,
    "min_samples_split",
    [2, 5, 10, 20, 40],
)
show_table(rf_split_df, "Random Forest — min_samples_split")
plot_sweep(
    rf_split_df,
    "Random Forest: min_samples_split",
    "min_samples_split",
)

parameter,value,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean
min_samples_split,2,0.9878,0.0116,0.9591,0.9435,0.8933
min_samples_split,5,0.9890,0.0098,0.9592,0.9435,0.9770
min_samples_split,10,0.9886,0.0106,0.9646,0.9517,0.9194
min_samples_split,20,0.9860,0.0113,0.9628,0.9486,0.9642
min_samples_split,40,0.9857,0.0108,0.9609,0.9454,0.9103


## 4. `min_samples_leaf`

This parameter directly controls terminal-region size.

Larger leaves act like smoothing:

$$
\text{larger leaves}
\Rightarrow
\text{less local variation}
\Rightarrow
\text{lower variance but potentially higher bias}.
$$

In [56]:
rf_leaf_df = run_sweep(
    rf_base,
    "min_samples_leaf",
    [1, 2, 4, 8, 16, 32],
)
show_table(rf_leaf_df, "Random Forest — min_samples_leaf")
plot_sweep(
    rf_leaf_df,
    "Random Forest: min_samples_leaf",
    "min_samples_leaf",
)

parameter,value,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean
min_samples_leaf,1,0.9878,0.0116,0.9591,0.9435,0.9676
min_samples_leaf,2,0.9871,0.0125,0.9630,0.9473,0.9016
min_samples_leaf,4,0.9866,0.0130,0.9665,0.9536,0.8667
min_samples_leaf,8,0.9850,0.0137,0.9611,0.9441,0.8534
min_samples_leaf,16,0.9845,0.0121,0.9554,0.9372,1.0253
min_samples_leaf,32,0.9830,0.0105,0.9479,0.9272,0.9138


## 5. `max_features`

This is one of Random Forest's defining parameters.

If every split considers every predictor, trees tend to become stronger but more correlated.

If only a small subset is considered, trees become more diverse but individually weaker.

The trade-off is:

$$
\boxed{
\text{tree strength}
\leftrightarrow
\text{tree correlation}
}
$$

In [57]:
rf_mf_df = run_sweep(
    rf_base,
    "max_features",
    [0.2, 0.5, "sqrt", "log2", 1.0],
)
show_table(rf_mf_df, "Random Forest — max_features")
plot_sweep(
    rf_mf_df,
    "Random Forest: max_features",
    "max_features",
    categorical=True,
)

parameter,value,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean
max_features,0.2,0.9865,0.0131,0.9609,0.9467,1.0103
max_features,0.5,0.9884,0.0102,0.9532,0.9379,0.9776
max_features,sqrt,0.9878,0.0116,0.9591,0.9435,0.9302
max_features,log2,0.9896,0.0096,0.9649,0.9492,0.9767
max_features,1.0,0.9887,0.0086,0.9586,0.9461,1.0246


## 6. Bootstrap sample fraction: `max_samples`

With `bootstrap=True`, this controls how much of the training set each tree samples.

Smaller fractions increase tree diversity.

But too little data weakens individual trees.

This is analogous to `max_features`, except the randomisation occurs along the **observation axis** rather than the feature axis.

In [58]:
rf_sample_df = run_sweep(
    rf_base,
    "max_samples",
    [0.4, 0.6, 0.8, 1.0],
)
show_table(rf_sample_df, "Random Forest — max_samples")
plot_sweep(
    rf_sample_df,
    "Random Forest: bootstrap sample fraction",
    "max_samples",
)

parameter,value,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean
max_samples,0.4,0.9870,0.0129,0.9666,0.9536,0.8123
max_samples,0.6,0.9874,0.0127,0.9631,0.9473,0.8012
max_samples,0.8,0.9877,0.0117,0.9646,0.9517,0.8100
max_samples,1.0,0.9878,0.0116,0.9591,0.9435,0.8318


## 7. Split criterion

We compare:

- Gini impurity
- entropy
- log-loss style impurity

These measure node heterogeneity differently, though in many datasets they lead to similar split choices.

In [59]:
rf_criterion_df = run_sweep(
    rf_base,
    "criterion",
    ["gini", "entropy", "log_loss"],
)
show_table(rf_criterion_df, "Random Forest — criterion")
plot_sweep(
    rf_criterion_df,
    "Random Forest: split criterion",
    "criterion",
    categorical=True,
)

parameter,value,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean
criterion,gini,0.9878,0.0116,0.9591,0.9435,0.8388
criterion,entropy,0.9903,0.0080,0.9594,0.9410,1.0373
criterion,log_loss,0.9903,0.0080,0.9594,0.9410,0.9246


## 8. `min_impurity_decrease`

This rejects splits whose improvement is too small.

Since the restriction is applied independently inside every tree, increasing it regularises the entire forest at the base-learner level.

In [60]:
rf_imp_df = run_sweep(
    rf_base,
    "min_impurity_decrease",
    [0.0, 1e-5, 1e-4, 1e-3, 1e-2],
)
show_table(rf_imp_df, "Random Forest — min_impurity_decrease")
plot_sweep(
    rf_imp_df,
    "Random Forest: min_impurity_decrease",
    "min_impurity_decrease",
)

parameter,value,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean
min_impurity_decrease,0.0000,0.9878,0.0116,0.9591,0.9435,0.9982
min_impurity_decrease,0.0000,0.9878,0.0116,0.9591,0.9435,0.8209
min_impurity_decrease,0.0001,0.9878,0.0117,0.9591,0.9435,0.8597
min_impurity_decrease,0.0010,0.9882,0.0111,0.9591,0.9435,0.9086
min_impurity_decrease,0.0100,0.9891,0.0081,0.9576,0.9378,0.9255


## 9. `max_leaf_nodes`

This places a direct upper bound on the number of terminal regions in each tree.

It is a useful alternative to `max_depth` when you want to reason in terms of the number of partition regions rather than the longest path.

In [61]:
rf_leaf_nodes_df = run_sweep(
    rf_base,
    "max_leaf_nodes",
    [4, 8, 16, 32, 64, None],
)
show_table(rf_leaf_nodes_df, "Random Forest — max_leaf_nodes")
plot_sweep(
    rf_leaf_nodes_df,
    "Random Forest: max_leaf_nodes",
    "max_leaf_nodes",
    categorical=True,
)

parameter,value,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean
max_leaf_nodes,4.0,0.9859,0.0123,0.9610,0.9454,1.1745
max_leaf_nodes,8.0,0.9862,0.0144,0.9612,0.9441,0.9333
max_leaf_nodes,16.0,0.9872,0.0120,0.9592,0.9435,0.8821
max_leaf_nodes,32.0,0.9875,0.0119,0.9592,0.9435,1.0342
max_leaf_nodes,64.0,0.9875,0.0119,0.9592,0.9435,1.0136
max_leaf_nodes,NaN,0.9878,0.0116,0.9591,0.9435,0.9226


## Random Forest decision-boundary comparison

We compare a strongly regularised forest, a moderate forest and a highly flexible forest.

In [62]:
rf_boundary_models = {
    "Shallow forest": RandomForestClassifier(
        n_estimators=250,
        max_depth=3,
        min_samples_leaf=8,
        max_features="sqrt",
        random_state=RANDOM_STATE,
    ),
    "Moderate forest": RandomForestClassifier(
        n_estimators=250,
        max_depth=7,
        min_samples_leaf=3,
        max_features="sqrt",
        random_state=RANDOM_STATE,
    ),
    "Highly flexible forest": RandomForestClassifier(
        n_estimators=250,
        max_depth=None,
        min_samples_leaf=1,
        max_features=1.0,
        random_state=RANDOM_STATE,
    ),
}

for name, model in rf_boundary_models.items():
    show_decision_boundary(
        model,
        f"{name} — decision probability surface",
    )

Two-moons test accuracy=0.914; ROC AUC=0.973


Two-moons test accuracy=0.929; ROC AUC=0.987


Two-moons test accuracy=0.924; ROC AUC=0.984


# Part V — Stacking

Stacking combines **heterogeneous model families** rather than averaging many versions of the same learner.

Suppose the base learners produce

$$
z_1(x), z_2(x), \ldots, z_M(x).
$$

A meta-model then learns

$$
\hat y = g(z_1(x), z_2(x), \ldots, z_M(x)).
$$

The key idea is **complementary error structure**.

For example:

- logistic regression contributes a stable linear view,
- SVM contributes a margin / kernel view,
- Random Forest contributes partition averaging,
- Gradient Boosting contributes sequential residual correction.

Stacking is useful only if those views contain complementary predictive information.

## Why out-of-fold meta-features are essential

A naive stacking implementation would fit a base learner on all training rows and then feed its predictions on those same rows to the meta-learner.

That leaks training fit.

Proper stacking instead creates **out-of-fold predictions**:

```text
fold 1 held out ← model trained on folds 2..K
fold 2 held out ← model trained on all other folds
...
fold K held out ← model trained on folds 1..K-1
```

The concatenated held-out predictions become meta-training features.

`StackingClassifier` performs this process internally.

In [31]:
stack_estimators = [
    (
        "logistic",
        Pipeline([
            ("scale", StandardScaler()),
            ("model", LogisticRegression(max_iter=3000)),
        ]),
    ),
    (
        "svm",
        Pipeline([
            ("scale", StandardScaler()),
            ("model", SVC(
                kernel="rbf",
                C=2.0,
                gamma="scale",
                probability=True,
                random_state=RANDOM_STATE,
            )),
        ]),
    ),
    (
        "rf",
        RandomForestClassifier(
            n_estimators=DEFAULT_RF_TREES,
            max_features="sqrt",
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
    ),
    (
        "gb",
        GradientBoostingClassifier(
            n_estimators=DEFAULT_GB_TREES,
            learning_rate=0.05,
            max_depth=2,
            random_state=RANDOM_STATE,
        ),
    ),
]

stack_model = StackingClassifier(
    estimators=stack_estimators,
    final_estimator=LogisticRegression(max_iter=3000),
    cv=CV_SPLITS,
    stack_method="predict_proba",
    passthrough=False,
    n_jobs=-1,
)

stack_model

StackingClassifier(cv=3,
                   estimators=[('logistic',
                                Pipeline(steps=[('scale', StandardScaler()),
                                                ('model',
                                                 LogisticRegression(max_iter=3000))])),
                               ('svm',
                                Pipeline(steps=[('scale', StandardScaler()),
                                                ('model',
                                                 SVC(C=2.0, probability=True,
                                                     random_state=42))])),
                               ('rf',
                                RandomForestClassifier(min_samples_leaf=2,
                                                       n_estimators=220,
                                                       n_jobs=-1,
                                                       random_state=42)),
                               ('gb',
                                GradientBoostingClassifier(learning_rate=0.05,
                                                           max_depth=2,
                                                           n_estimators=180,
                                                           random_state=42))],
                   final_estimator=LogisticRegression(max_iter=3000), n_jobs=-1,
                   stack_method='predict_proba')

## Base learners versus stack

A stack should not automatically be declared superior because it is more sophisticated.

It adds:

- more models,
- greater fit time,
- more memory,
- more complex deployment,
- harder debugging and monitoring.

Therefore the relevant question is:

> Does the cross-validated gain justify the operational cost?

In [32]:
stack_compare_models = {
    "Logistic": stack_estimators[0][1],
    "RBF SVM": stack_estimators[1][1],
    "Random Forest": stack_estimators[2][1],
    "Gradient Boosting": stack_estimators[3][1],
    "Stacking": stack_model,
}

stack_rows = []

for name, model in stack_compare_models.items():
    stack_rows.append({
        "model": name,
        **cv_evaluate(model),
    })

stack_df = (
    pd.DataFrame(stack_rows)
    .sort_values("roc_auc_mean", ascending=False)
    .reset_index(drop=True)
)

show_table(
    stack_df,
    "Stacking versus component models",
    height=260,
)

model,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean
Logistic,0.9953,0.0045,0.9816,0.9711,0.0106
RBF SVM,0.9943,0.0050,0.9758,0.9668,0.0129
Stacking,0.9940,0.0049,0.9798,0.9692,4.2253
Gradient Boosting,0.9876,0.0071,0.9571,0.9417,0.6582
Random Forest,0.9871,0.0125,0.9630,0.9473,0.8666


In [33]:
p = figure(
    x_range=list(stack_df["model"]),
    width=900,
    height=420,
    title="Stacking comparison — cross-validated ROC AUC",
    x_axis_label="model",
    y_axis_label="mean ROC AUC",
)

p.vbar(
    x=stack_df["model"],
    top=stack_df["roc_auc_mean"],
    width=0.65,
)

p.xaxis.major_label_orientation = 0.7
show(p)

## `passthrough=True`

With `passthrough=False`, the meta-model sees only base-model predictions.

With `passthrough=True`, it also receives the original features.

This increases meta-model capacity.

Potential benefit:

- raw features can help correct systematic weaknesses of the base learners.

Potential cost:

- more dimensions,
- greater overfitting risk,
- reduced conceptual separation between base and meta levels.

In [34]:
passthrough_rows = []

for passthrough in [False, True]:
    model = clone(stack_model).set_params(passthrough=passthrough)
    passthrough_rows.append({
        "passthrough": passthrough,
        **cv_evaluate(model),
    })

show_table(
    pd.DataFrame(passthrough_rows),
    "Stacking — passthrough comparison",
    height=180,
)

passthrough,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean
False,0.9940,0.0049,0.9798,0.9692,4.8295
True,0.9952,0.0026,0.9663,0.9536,4.2617


# Part VI — SVM beyond $C$

An SVM is often taught mainly through the regularisation parameter $C$.

That is incomplete.

For nonlinear SVMs, the **kernel and kernel parameters define the geometry of similarity**.

We investigate:

1. kernel family,
2. $\gamma$,
3. polynomial degree,
4. `coef0`,
5. shrinking heuristic,
6. optimisation tolerance.

In [35]:
svm_base = Pipeline([
    ("scale", StandardScaler()),
    ("svc", SVC(
        kernel="rbf",
        C=2.0,
        gamma="scale",
        probability=True,
        shrinking=True,
        tol=1e-3,
        random_state=RANDOM_STATE,
    )),
])

svm_base

Pipeline(steps=[('scale', StandardScaler()),
                ('svc', SVC(C=2.0, probability=True, random_state=42))])

## 1. Kernel family

### Linear

$$
K(x_i,x_j)=x_i^Tx_j.
$$

### Polynomial

$$
K(x_i,x_j)
=
(\gamma x_i^Tx_j+r)^d.
$$

### RBF

$$
K(x_i,x_j)
=
\exp(-\gamma\|x_i-x_j\|^2).
$$

### Sigmoid

$$
K(x_i,x_j)
=
\tanh(\gamma x_i^Tx_j+r).
$$

Kernel choice determines what kinds of boundaries are easy for the SVM to express.

In [36]:
svm_kernel_df = run_sweep(
    svm_base,
    "svc__kernel",
    ["linear", "poly", "rbf", "sigmoid"],
)
show_table(svm_kernel_df, "SVM — kernel comparison")
plot_sweep(
    svm_kernel_df,
    "SVM: kernel family",
    "kernel",
    categorical=True,
)

parameter,value,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean
svc__kernel,linear,0.9929,0.0037,0.9796,0.9692,0.0110
svc__kernel,poly,0.9924,0.0044,0.9321,0.8774,0.0204
svc__kernel,rbf,0.9943,0.0050,0.9758,0.9668,0.0119
svc__kernel,sigmoid,0.9842,0.0145,0.9598,0.9397,0.0096


## 2. $\gamma$ — locality of RBF similarity

For

$$
K(x_i,x_j)
=
\exp(-\gamma\|x_i-x_j\|^2),
$$

small $\gamma$ means similarity decays slowly.

Each observation influences a broad region, producing a smoother decision surface.

Large $\gamma$ means similarity decays rapidly.

Influence becomes highly local and the boundary can become much more flexible.

Therefore:

$$
\gamma\uparrow
\Rightarrow
\text{locality}\uparrow
\Rightarrow
\text{effective complexity}\uparrow.
$$

In [37]:
svm_gamma_df = run_sweep(
    svm_base,
    "svc__gamma",
    [0.001, 0.005, 0.01, 0.05, 0.2, 1.0],
)
show_table(svm_gamma_df, "RBF SVM — gamma")
plot_sweep(
    svm_gamma_df,
    "RBF SVM: gamma",
    "gamma",
)

parameter,value,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean
svc__gamma,0.001,0.9921,0.0064,0.9642,0.9371,0.0180
svc__gamma,0.005,0.9942,0.0050,0.9780,0.9648,0.0134
svc__gamma,0.010,0.9943,0.0050,0.9724,0.9579,0.0121
svc__gamma,0.050,0.9922,0.0050,0.9758,0.9668,0.0142
svc__gamma,0.200,0.9763,0.0124,0.9425,0.9324,0.0310
svc__gamma,1.000,0.9488,0.0125,0.7739,0.5094,0.0386


## 3. Polynomial degree

The polynomial kernel is

$$
K(x_i,x_j)
=
(\gamma x_i^Tx_j+\text{coef0})^d.
$$

Increasing $d$ allows higher-order interactions.

That can increase expressive power, but also sensitivity, variance and computational difficulty.

In [38]:
poly_base = clone(svm_base).set_params(
    svc__kernel="poly",
    svc__gamma="scale",
    svc__coef0=1.0,
)

svm_degree_df = run_sweep(
    poly_base,
    "svc__degree",
    [2, 3, 4, 5, 6],
)
show_table(svm_degree_df, "Polynomial SVM — degree")
plot_sweep(
    svm_degree_df,
    "Polynomial SVM: degree",
    "degree",
)

parameter,value,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean
svc__degree,2,0.9965,0.0049,0.9796,0.9705,0.0132
svc__degree,3,0.9958,0.0058,0.9778,0.9674,0.0116
svc__degree,4,0.9941,0.0066,0.9759,0.9655,0.0085
svc__degree,5,0.9919,0.0057,0.9705,0.9560,0.0110
svc__degree,6,0.9904,0.0050,0.9706,0.9548,0.0105


## 4. `coef0`

`coef0` controls the independent offset inside polynomial and sigmoid kernels.

For the polynomial kernel:

$$
K(x_i,x_j)
=
(\gamma x_i^Tx_j+\text{coef0})^d.
$$

Changing `coef0` changes the balance among lower-order and higher-order terms after expansion.

It is therefore not merely a cosmetic intercept.

In [39]:
svm_coef_df = run_sweep(
    poly_base,
    "svc__coef0",
    [0.0, 0.25, 0.5, 1.0, 2.0, 5.0],
)
show_table(svm_coef_df, "Polynomial SVM — coef0")
plot_sweep(
    svm_coef_df,
    "Polynomial SVM: coef0",
    "coef0",
)

parameter,value,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean
svc__coef0,0.00,0.9924,0.0044,0.9321,0.8774,0.0145
svc__coef0,0.25,0.9948,0.0027,0.9707,0.9535,0.0104
svc__coef0,0.50,0.9952,0.0039,0.9760,0.9642,0.0137
svc__coef0,1.00,0.9958,0.0058,0.9778,0.9674,0.0084
svc__coef0,2.00,0.9948,0.0061,0.9778,0.9674,0.0094
svc__coef0,5.00,0.9928,0.0061,0.9681,0.9580,0.0092


## 5. Shrinking heuristic

The SVM solver can heuristically remove variables that appear unlikely to become support vectors during part of the optimisation.

`shrinking` is therefore mainly an **optimisation parameter**, not a direct statistical complexity parameter.

We expect:

- predictive performance to remain similar,
- fit time potentially to change.

In [40]:
svm_shrink_df = run_sweep(
    svm_base,
    "svc__shrinking",
    [True, False],
)
show_table(svm_shrink_df, "SVM — shrinking heuristic")
plot_sweep(
    svm_shrink_df,
    "SVM: shrinking heuristic",
    "shrinking",
    categorical=True,
)

parameter,value,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean
svc__shrinking,True,0.9943,0.005,0.9758,0.9668,0.0170
svc__shrinking,False,0.9943,0.005,0.9758,0.9668,0.0109


## 6. Optimisation tolerance

`tol` determines how precisely the numerical optimiser must converge.

Smaller tolerance:

- seeks a more precise optimum,
- may take longer,
- often produces little predictive improvement after convergence is already adequate.

This is another example of a parameter whose main effect may be computational rather than statistical.

In [41]:
svm_tol_df = run_sweep(
    svm_base,
    "svc__tol",
    [1e-2, 3e-3, 1e-3, 3e-4, 1e-4],
)
show_table(svm_tol_df, "SVM — optimisation tolerance")
plot_sweep(
    svm_tol_df,
    "SVM: optimisation tolerance",
    "tol",
)

parameter,value,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean
svc__tol,0.0100,0.9943,0.005,0.9758,0.9668,0.0136
svc__tol,0.0030,0.9943,0.005,0.9758,0.9668,0.0113
svc__tol,0.0010,0.9943,0.005,0.9758,0.9668,0.0116
svc__tol,0.0003,0.9943,0.005,0.9758,0.9668,0.0104
svc__tol,0.0001,0.9943,0.005,0.9758,0.9668,0.0133


# Part VII — SVM decision boundaries

We now compare multiple kernels on exactly the same two-moons split.

This shows how the kernel changes the geometry even when $C$ is held approximately comparable.

In [42]:
svm_boundary_models = {
    "Linear kernel": Pipeline([
        ("scale", StandardScaler()),
        ("svc", SVC(
            kernel="linear",
            C=2.0,
            probability=True,
            random_state=RANDOM_STATE,
        )),
    ]),
    "Polynomial kernel": Pipeline([
        ("scale", StandardScaler()),
        ("svc", SVC(
            kernel="poly",
            degree=3,
            C=2.0,
            gamma="scale",
            coef0=1.0,
            probability=True,
            random_state=RANDOM_STATE,
        )),
    ]),
    "RBF kernel": Pipeline([
        ("scale", StandardScaler()),
        ("svc", SVC(
            kernel="rbf",
            C=2.0,
            gamma=1.0,
            probability=True,
            random_state=RANDOM_STATE,
        )),
    ]),
    "Sigmoid kernel": Pipeline([
        ("scale", StandardScaler()),
        ("svc", SVC(
            kernel="sigmoid",
            C=2.0,
            gamma="scale",
            coef0=0.0,
            probability=True,
            random_state=RANDOM_STATE,
        )),
    ]),
}

for name, model in svm_boundary_models.items():
    show_decision_boundary(
        model,
        f"SVM — {name}",
    )

Two-moons test accuracy=0.862; ROC AUC=0.951


Two-moons test accuracy=0.933; ROC AUC=0.986


Two-moons test accuracy=0.948; ROC AUC=0.987


Two-moons test accuracy=0.695; ROC AUC=0.823


## RBF $\gamma$ boundary experiment

We hold $C$ fixed and vary only $\gamma$.

This visualises the transition from broad, smooth influence to highly local influence.

In [43]:
for gamma in [0.05, 0.5, 5.0]:
    model = Pipeline([
        ("scale", StandardScaler()),
        ("svc", SVC(
            kernel="rbf",
            C=2.0,
            gamma=gamma,
            probability=True,
            random_state=RANDOM_STATE,
        )),
    ])

    show_decision_boundary(
        model,
        f"RBF SVM — gamma={gamma}",
    )

Two-moons test accuracy=0.876; ROC AUC=0.953


Two-moons test accuracy=0.924; ROC AUC=0.988


Two-moons test accuracy=0.943; ROC AUC=0.972


# Part VIII — Unified benchmark

We now compare representative versions of:

- Logistic Regression
- Random Forest
- Gradient Boosting
- AdaBoost
- Histogram Gradient Boosting
- RBF SVM
- Stacking

under the same stratified cross-validation design.

In [44]:
benchmark_models = {
    "Logistic Regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=3000)),
    ]),
    "Random Forest": rf_base,
    "Gradient Boosting": gb_base,
    "AdaBoost": boost_models["AdaBoost"],
    "Hist Gradient Boosting": boost_models["Hist Gradient Boosting"],
    "RBF SVM": svm_base,
    "Stacking": stack_model,
}

benchmark_rows = []

for name, model in benchmark_models.items():
    benchmark_rows.append({
        "model": name,
        **cv_evaluate(model),
    })

benchmark_df = (
    pd.DataFrame(benchmark_rows)
    .sort_values("roc_auc_mean", ascending=False)
    .reset_index(drop=True)
)

show_table(
    benchmark_df,
    "Unified cross-validated benchmark",
    height=320,
)

model,roc_auc_mean,roc_auc_sd,f1_mean,balanced_accuracy_mean,fit_time_mean
Logistic Regression,0.9953,0.0045,0.9816,0.9711,0.0078
RBF SVM,0.9943,0.0050,0.9758,0.9668,0.0110
Stacking,0.9940,0.0049,0.9798,0.9692,3.6099
Hist Gradient Boosting,0.9883,0.0084,0.9644,0.9530,0.2109
Random Forest,0.9878,0.0116,0.9591,0.9435,0.7854
Gradient Boosting,0.9876,0.0071,0.9571,0.9417,0.5904
AdaBoost,0.9864,0.0079,0.9572,0.9404,0.5604


In [45]:
p = figure(
    x_range=list(benchmark_df["model"]),
    width=950,
    height=430,
    title="Unified benchmark — ROC AUC",
    x_axis_label="model",
    y_axis_label="cross-validated ROC AUC",
)

p.vbar(
    x=benchmark_df["model"],
    top=benchmark_df["roc_auc_mean"],
    width=0.65,
)

p.xaxis.major_label_orientation = 0.75
show(p)

# Part IX — Hyperparameter interpretation summary

## Gradient Boosting

| Hyperparameter | Main role | Increasing it usually means |
|---|---|---|
| `learning_rate` | stage size | more aggressive corrections |
| `n_estimators` | number of stages | more additive capacity |
| `subsample` | row fraction per stage | less randomness as it approaches 1 |
| `max_depth` | base-tree complexity | more interactions per stage |
| `min_impurity_decrease` | split threshold | stronger regularisation |
| `max_features` | feature randomness | less randomness when more features are used |
| `max_leaf_nodes` | terminal regions | more complex base trees |

## Random Forest

| Hyperparameter | Main role | Increasing it usually means |
|---|---|---|
| `n_estimators` | ensemble size | more stable averaging |
| `max_depth` | tree capacity | lower bias, higher tree variance |
| `min_samples_split` | split regularisation | fewer small-node splits |
| `min_samples_leaf` | leaf smoothing | larger, more stable leaves |
| `max_features` | feature availability | stronger but more correlated trees |
| `max_samples` | bootstrap fraction | stronger but more similar trees |
| `min_impurity_decrease` | split threshold | stronger regularisation |
| `max_leaf_nodes` | tree capacity | more terminal regions |

## SVM

| Hyperparameter | Main role |
|---|---|
| kernel | geometry of similarity |
| $C$ | margin-violation penalty |
| $\gamma$ | locality for RBF/poly/sigmoid |
| degree | polynomial interaction order |
| `coef0` | independent offset inside poly/sigmoid kernel |
| shrinking | optimisation heuristic |
| tolerance | convergence precision |

The goal is to understand **mechanism**, not memorise a tuning grid.

# Part X — Important cautions

## 1. One-at-a-time sweeps are diagnostic

The best value of one parameter may depend strongly on another.

Important interactions include:

$$
\text{learning rate}\times\text{n_estimators}
$$

$$
\text{max_depth}\times\text{min_samples_leaf}
$$

$$
C\times\gamma
$$

A real tuning workflow should move from interpretable sweeps to joint optimisation.

## 2. Never tune on the final test set

Cross-validation belongs inside model development. The test set should remain untouched until major modelling choices are settled.

## 3. More sophisticated does not always mean better

A stack that improves ROC AUC from 0.992 to 0.993 but doubles latency may not be the correct engineering choice.

## 4. Decision boundaries are intuition devices

A visually appealing two-dimensional boundary does not prove superiority in a 30-dimensional real dataset.

# Part XI — Student exercises

## Exercise 1 — Three-way Gradient Boosting interaction

Study:

- learning rate,
- number of estimators,
- max depth.

**Question:** Does the preferred depth change when the learning rate becomes smaller?

---

## Exercise 2 — Stochastic boosting

Cross learning rate with

$$
\text{subsample}\in\{0.4,0.6,0.8,1.0\}.
$$

**Question:** Does subsampling help more when individual updates are aggressive?

---

## Exercise 3 — Random Forest diversity

Jointly vary:

- `max_features`,
- `max_samples`.

**Question:** At what point does additional randomness weaken the trees enough to hurt the forest?

---

## Exercise 4 — OOB versus cross-validation

Enable:

```python
oob_score=True
```

Compare:

- OOB score,
- cross-validation score,
- final held-out score.

Explain why they need not match exactly.

---

## Exercise 5 — Stacking diversity

Create one stack containing several very similar forests and another containing heterogeneous learners.

**Question:** Why can a smaller but more diverse stack outperform a larger redundant stack?

---

## Exercise 6 — Meta-model flexibility

Replace logistic regression as the meta-model with:

- Random Forest
- Gradient Boosting

**Question:** Does a more powerful meta-model improve cross-validation performance or mostly increase overfitting risk?

---

## Exercise 7 — SVM $C\times\gamma$

Construct a heatmap over:

$$
C\in\{0.1,1,10,100\}
$$

and

$$
\gamma\in\{0.001,0.01,0.1,1\}.
$$

Identify the high-bias and high-variance regions.

---

## Exercise 8 — Polynomial SVM

Jointly vary:

- degree,
- `coef0`,
- $\gamma$.

**Question:** Which parameter changes the decision geometry most strongly?

---

## Exercise 9 — Statistical versus optimisation parameters

Compare the performance effect and fit-time effect of:

- $\gamma$,
- `shrinking`,
- `tol`.

Classify each as mainly statistical, mainly computational, or both.

---

## Exercise 10 — Engineering decision

Suppose stacking produces only a tiny AUC gain but doubles prediction latency.

Discuss:

- cost of false negatives,
- latency requirements,
- probability calibration,
- model monitoring,
- operational complexity,
- uncertainty of the measured gain.

# Final synthesis

Several deeper principles connect these experiments.

### 1. `n_estimators` does not mean the same thing everywhere

For Random Forest it mainly stabilises an average.

For Gradient Boosting it expands a sequential additive model.

### 2. Randomisation can act as regularisation

Examples include:

- Gradient Boosting `subsample`
- Random Forest `max_features`
- Random Forest `max_samples`

### 3. Tree complexity can be controlled in multiple ways

- depth
- minimum leaf size
- minimum split size
- impurity threshold
- maximum leaf nodes

Each restricts the partition of feature space differently.

### 4. Stacking exploits diversity, not model count

The meta-learner benefits when component models make complementary errors.

### 5. SVM complexity is geometric

$C$ matters, but kernel choice, $\gamma$, degree and `coef0` determine the family of similarities and decision surfaces the model can express.

The common thread is:

$$
\boxed{
\text{Hyperparameters encode assumptions about complexity, locality, randomness and optimisation.}
}
$$